# Danh gia bo du lieu Website du lich LSA

In [1]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()
import shutil

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
import os
import re
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

In [3]:
from nltk.tokenize import sent_tokenize

def split_sentences(text):
    return sent_tokenize(text)

In [4]:
from nltk.tokenize import sent_tokenize
from pyvi import ViTokenizer

def tokenize_vi_sentence_level(text: str) -> list[str]:
    sentences = sent_tokenize(text)
    tokens = []

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        sent_tokens = ViTokenizer.tokenize(sent)
        tokens.extend(sent_tokens.split())

    return tokens


In [5]:
import re

VI_TOKEN_REGEX = re.compile(
    r"[a-zàáạảãâầấậẩẫăằắặẳẵ"
    r"èéẹẻẽêềếệểễ"
    r"ìíịỉĩ"
    r"òóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữ"
    r"ỳýỵỷỹđ0-9_]+$"
)

def is_valid_vi_token(token: str) -> bool:
    return bool(VI_TOKEN_REGEX.fullmatch(token))


In [6]:
def load_stopwords(path):
    with open(path, "r", encoding="utf-8") as f:
        stopwords = set(
            line.strip().lower()
            for line in f
            if line.strip()
        )
    return stopwords

STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

print(f"🛑 Đã load {len(vi_stopwords)} stopword")

🛑 Đã load 1942 stopword


In [7]:
# === Đọc dữ liệu và tiền xử lý ===
import re
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    
    # Xóa URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # text = text.lower()

    # # Loại ký tự không cần thiết (giữ chữ, số, dấu câu cơ bản)
    # text = re.sub(r"[^0-9a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễ"
    #               r"ìíịỉĩòóọỏõôồốộổỗơờớợởỡ"
    #               r"ùúụủũưừứựửữỳýỵỷỹđ\s.,!?]", " ", text)

    # Chuẩn hóa dấu câu
    text = re.sub(r"[.,!?]+", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [38]:
def preprocess_query(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)
    print(tokens)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [39]:
STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

query = "Những địa điểm du lịch nổi tiếng nhất ở Hà Nội là gì?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

['Những', 'địa_điểm', 'du_lịch', 'nổi_tiếng', 'nhất', 'ở', 'Hà_Nội', 'là', 'gì']
địa_điểm du_lịch nổi_tiếng hà_nội


In [40]:
query = "Cao Bằng có những di tích lịch sử nào?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

['Cao', 'Bằng', 'có', 'những', 'di_tích', 'lịch_sử', 'nào']
di_tích lịch_sử


In [41]:
query = "Quảng trường nào nổi tiếng tại TP. Hồ Chí Minh?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

['Quảng_trường', 'nào', 'nổi_tiếng', 'tại', 'TP', 'Hồ_Chí_Minh']
quảng_trường nổi_tiếng tp hồ_chí_minh


In [ ]:
query = "Nhà thờ nào nổi tiếng để chụp ảnh ở Đà Lạt?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

['Nhà_thờ', 'nào', 'nổi_tiếng', 'để', 'chụp', 'ảnh', 'ở', 'Đà_Lạt']
nhà_thờ nổi_tiếng chụp ảnh đà_lạt


In [43]:
query = "Núi nào cao nhất Việt Nam?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

['Núi', 'nào', 'cao', 'nhất', 'Việt_Nam']
núi việt_nam


In [44]:
query = "Vịnh biển nào nổi tiếng ở miền Bắc Việt Nam?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

['Vịnh', 'biển', 'nào', 'nổi_tiếng', 'ở', 'miền', 'Bắc', 'Việt_Nam']
vịnh biển nổi_tiếng miền bắc việt_nam


In [10]:
import re

def preprocess(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    tokens = [t.lower() for t in tokens if len(t) > 1]
    return " ".join(tokens)


In [11]:
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer

def create_schema():
    return Schema(
        docid=ID(stored=True, unique=True),
        title=TEXT(stored=True, analyzer=StandardAnalyzer()),
        content=TEXT(stored=True, analyzer=StandardAnalyzer())
    )


In [12]:
import shutil
import os

def build_index(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = row["title"]
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [21]:
def readQuery(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query(row["query"], vi_stopwords)

    return queries


In [22]:
import ast

def readGroundTruth(query_csv, meta_csv):
    meta = pd.read_csv(meta_csv)
    url2docid = dict(zip(meta["url"], meta["id"]))

    df = pd.read_csv(query_csv)
    qrels = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        qrels[qid] = {}

        urls = ast.literal_eval(row["urls"])
        for u in urls:
            if u in url2docid:
                qrels[qid][str(url2docid[u])] = 1

    return qrels


In [23]:
from gensim import corpora, models

def build_lsa(ix, num_topics=200):
    texts = []
    docids = []

    with ix.searcher() as searcher:
        for doc in searcher.all_stored_fields():
            tokens = doc["content"].split()
            texts.append(tokens)
            docids.append(doc["docid"])

    dictionary = corpora.Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]

    tfidf = models.TfidfModel(corpus)
    corpus_tfidf = tfidf[corpus]

    lsa = models.LsiModel(
        corpus_tfidf,
        id2word=dictionary,
        num_topics=num_topics
    )

    corpus_lsa = lsa[corpus_tfidf]

    return dictionary, tfidf, lsa, corpus_lsa, docids


In [24]:
from gensim.similarities import MatrixSimilarity

def lsa_search(query, dictionary, tfidf, lsa, corpus_lsa, docids, top_k=100):
    vec = dictionary.doc2bow(query.split())
    vec_lsa = lsa[tfidf[vec]]

    index = MatrixSimilarity(corpus_lsa)
    sims = index[vec_lsa]

    ranked = sorted(enumerate(sims), key=lambda x: -x[1])[:top_k]

    results = {}
    for rank, (idx, score) in enumerate(ranked):
        results[str(docids[idx])] = float(score)

    return results


In [25]:
def run_lsa_all_queries(queries, lsa_components):
    dictionary, tfidf, lsa, corpus_lsa, docids = lsa_components
    run = {}

    for qid, query in queries.items():
        run[qid] = lsa_search(
            query,
            dictionary,
            tfidf,
            lsa,
            corpus_lsa,
            docids
        )

    return run


In [26]:
def evaluate_set_retrieval_at_k(GroundTruth, RunResults, cutoffs=[5,10,20]):
    """
    GroundTruth: dict {qid: {docid: relevance}}
    RunResults : dict {qid: {docid: score}}
    """

    per_query = {}
    avg_metrics = {k: {"P":0, "R":0, "F1":0} for k in cutoffs}
    n = len(GroundTruth)

    print("========== Per-query results ==========")

    for qid in GroundTruth:
        relevant = set(GroundTruth[qid].keys())

        ranked_docs = sorted(
            RunResults.get(qid, {}).items(),
            key=lambda x: x[1],
            reverse=True
        )

        per_query[qid] = {}

        for k in cutoffs:
            retrieved_k = set(docid for docid, _ in ranked_docs[:k])

            tp = len(relevant & retrieved_k)
            fp = len(retrieved_k) - tp
            fn = len(relevant) - tp

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

            per_query[qid][f"P@{k}"] = precision
            per_query[qid][f"R@{k}"] = recall
            per_query[qid][f"F1@{k}"] = f1

            avg_metrics[k]["P"] += precision
            avg_metrics[k]["R"] += recall
            avg_metrics[k]["F1"] += f1

        print(f"Query {qid}")
        for k in cutoffs:
            print(f"  @ {k}")
            print(f"    Precision : {per_query[qid][f'P@{k}']:.4f}")
            print(f"    Recall    : {per_query[qid][f'R@{k}']:.4f}")
            print(f"    F1        : {per_query[qid][f'F1@{k}']:.4f}")
        print("-" * 30)

    print("\n========== Average over all queries ==========")
    for k in cutoffs:
        print(f"@{k}")
        print(f"  Precision : {avg_metrics[k]['P']/n:.4f}")
        print(f"  Recall    : {avg_metrics[k]['R']/n:.4f}")
        print(f"  F1        : {avg_metrics[k]['F1']/n:.4f}")

    return per_query, avg_metrics


## Bo cac tu it xuat hien

In [27]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã xây dựng xong LSA
✅ Đã chạy xong tất cả truy vấn


In [28]:
queries

{'1': 'nhà_thờ kon_tum',
 '2': 'vũng_tàu địa_điểm đẹp',
 '3': 'địa_điểm du_lịch nổi_tiếng hà_nội',
 '4': 'đi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an trải nghiệm đặc_sắc',
 '6': 'lý_tưởng du_lịch sa_pa',
 '7': 'tham_quan lỡ huế',
 '8': 'du_lịch ninh_bình đi tràng_an tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật miền tây_nam_bộ',
 '10': 'check in đẹp đà_lạt giới trẻ',
 '11': 'du_lịch hạ_long tour hoạt_động hấp_dẫn',
 '12': 'địa_điểm du_lịch tâm_linh nổi_tiếng việt_nam',
 '13': 'đi du_lịch côn_đảo mùa',
 '14': 'địa_điểm du_lịch tp hcm đi tuần',
 '15': 'du_lịch mộc_châu hấp_dẫn mùa hoa',
 '16': 'vườn quốc_gia đẹp nổi_tiếng việt_nam',
 '17': 'du_lịch quy_nhơn bãi biển hoang_sơ',
 '18': 'du_lịch miền huế đi',
 '19': 'đồng_nai núi',
 '20': 'chùa nổi_tiếng hà_nội',
 '21': 'núi nổi_tiếng leo sa_pa lào_cai',
 '22': 'thác đẹp nổi_tiếng đà_lạt',
 '23': 'hang_động nổi_tiếng quảng_bình',
 '24': 'di_tích lịch_sử nổi_bật cố_đô huế',
 '25': 'cầu nổi_tiếng check in đà_nẵng',
 '26': 'làng_nghề truy

In [105]:
qrels

{'1': {'494': 1, '495': 1},
 '2': {'9': 1,
  '123': 1,
  '241': 1,
  '242': 1,
  '424': 1,
  '425': 1,
  '426': 1,
  '427': 1},
 '3': {'49': 1,
  '350': 1,
  '351': 1,
  '356': 1,
  '357': 1,
  '358': 1,
  '359': 1,
  '470': 1,
  '471': 1},
 '4': {'37': 1,
  '279': 1,
  '185': 1,
  '280': 1,
  '281': 1,
  '285': 1,
  '286': 1,
  '287': 1,
  '288': 1,
  '289': 1,
  '290': 1,
  '484': 1,
  '485': 1,
  '486': 1,
  '487': 1},
 '5': {'51': 1, '129': 1, '130': 1, '131': 1, '304': 1, '305': 1},
 '6': {'100': 1,
  '188': 1,
  '189': 1,
  '381': 1,
  '382': 1,
  '383': 1,
  '384': 1,
  '462': 1,
  '463': 1,
  '464': 1,
  '465': 1},
 '7': {'114': 1,
  '128': 1,
  '248': 1,
  '249': 1,
  '352': 1,
  '353': 1,
  '354': 1,
  '355': 1,
  '385': 1,
  '386': 1,
  '436': 1,
  '437': 1},
 '8': {'134': 1,
  '135': 1,
  '112': 1,
  '78': 1,
  '246': 1,
  '247': 1,
  '348': 1,
  '349': 1},
 '9': {'399': 1,
  '400': 1,
  '406': 1,
  '407': 1,
  '408': 1,
  '428': 1,
  '429': 1,
  '430': 1,
  '431': 1,
  '43

In [29]:
run

{'1': {'494': 0.9244186282157898,
  '362': 0.6657742261886597,
  '363': 0.6263532042503357,
  '57': 0.5603055953979492,
  '314': 0.37380579113960266,
  '495': 0.3424055576324463,
  '67': 0.3395121395587921,
  '232': 0.20903177559375763,
  '78': 0.14437147974967957,
  '172': 0.12832093238830566,
  '333': 0.12815771996974945,
  '205': 0.12385158240795135,
  '201': 0.12363301217556,
  '405': 0.11817015707492828,
  '207': 0.1145377904176712,
  '297': 0.10963600873947144,
  '372': 0.10837000608444214,
  '368': 0.09713627398014069,
  '349': 0.09557878971099854,
  '216': 0.09476779401302338,
  '410': 0.09064622968435287,
  '328': 0.08621810376644135,
  '390': 0.08477199077606201,
  '331': 0.08109981566667557,
  '400': 0.07678954303264618,
  '167': 0.07167547196149826,
  '221': 0.06727379560470581,
  '49': 0.06638918817043304,
  '165': 0.06484159827232361,
  '75': 0.06462226808071136,
  '445': 0.06355305761098862,
  '177': 0.06341753900051117,
  '5': 0.06227225065231323,
  '235': 0.06131225824

In [35]:
import pytrec_eval
import math

def print_best_worst_queries(
    RunResults,
    Queries,
    GroundTruth,
    top_k=5,
    retrieved_k=10
):
    evaluator = pytrec_eval.RelevanceEvaluator(
        GroundTruth,
        {"map", "P_10"}
    )
    results = evaluator.evaluate(RunResults)

    # Lọc query hợp lệ
    valid = [
        (qid, res["map"], res["P_10"])
        for qid, res in results.items()
        if not math.isnan(res["map"])
    ]

    # Sort theo MAP (bạn có thể đổi sang P@10 nếu muốn)
    valid_sorted = sorted(valid, key=lambda x: x[1], reverse=True)

    best = valid_sorted[:top_k]
    worst = valid_sorted[-top_k:]

    def print_block(title, items):
        print("\n" + "=" * 60)
        print(title)
        print("=" * 60)

        for qid, map_score, p10_score in items:
            print(f"\nQuery ID : {qid}")
            print(f"Query    : {Queries[qid]}")
            print(f"MAP      : {map_score:.4f}")
            print(f"P@10     : {p10_score:.4f}")

            # Relevant docs
            rel_docs = [
                docid for docid, rel in GroundTruth[qid].items()
                if rel > 0
            ]
            print(f"Relevant docs ({len(rel_docs)}): {rel_docs[:retrieved_k]}")

            # Retrieved docs
            retrieved = list(RunResults[qid].keys())[:retrieved_k]
            print(f"Top retrieved docs: {retrieved}")

    print_block("🔥 TOP QUERIES (Highest MAP)", best)
    print_block("❄️ WORST QUERIES (Lowest MAP)", worst)


In [36]:
print_best_worst_queries(run, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 11
Query    : du_lịch hạ_long tour hoạt_động hấp_dẫn
MAP      : 1.0000
P@10     : 0.4000
Relevant docs (4): ['50', '122', '132', '133']
Top retrieved docs: ['133', '50', '122', '132', '93', '229', '203', '121', '168', '338']

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['30', '218', '169', '168', '376', '331', '119', '377', '216', '349']

Query ID : 27
Query    : chợ đêm nổi_tiếng đông đà_lạt
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['143', '38']
Top retrieved docs: ['38', '143', '363', '217', '172', '275', '362', '365', '318', '216']

Query ID : 33
Query    : đèo đẹp nổi_tiếng phượt miền trung
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['436', '437']
Top retrieved docs: ['437', '436', '100', '37', '64', '189', '61', '493', '68', '52']

Query ID : 37
Query    : thành nhà_hồ
MAP      : 1.0000
P@10     : 0.1000
Relevant docs (1): ['151']
To

In [30]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.3500
    Recall    : 0.8750
    F1        : 0.5000
------------------------------
Query 4
  @ 5
    Precision : 1.0000
    Recall    : 0.4167
    F1        : 0.5882
  @ 10
    Precision : 1.0000
    Recall    : 0.8333
    F1        : 0.9091
  @

In [5]:
import gc
import time
import os
import shutil

def safe_remove_index(index_dir):
    try:
        gc.collect()
        time.sleep(1)
        shutil.rmtree(index_dir)
    except PermissionError as e:
        print("⚠️ Không xóa được index, hãy restart kernel:", e)


In [6]:
INDEX_DIR="ind"
if os.path.exists(INDEX_DIR):
    safe_remove_index(INDEX_DIR)

⚠️ Không xóa được index, hãy restart kernel: [WinError 32] The process cannot access the file because it is being used by another process: 'ind\\MAIN_ol2gz8tuxhf04ota.seg'


## Chi bo stopword

In [46]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
['Nhà_thờ', 'ở', 'Kon_Tum']
['Vũng_Tàu', 'có', 'các', 'địa_điểm', 'nào', 'đẹp']
['Những', 'địa_điểm', 'du_lịch', 'nổi_tiếng', 'nhất', 'ở', 'Hà_Nội', 'là', 'gì']
['Nên', 'đi', 'đâu', 'khi', 'du_lịch', 'Đà_Nẵng']
['Du_lịch', 'Hội_An', 'có', 'những', 'trải', 'nghiệm', 'đặc_sắc', 'nào']
['Thời_điểm', 'lý_tưởng', 'để', 'du_lịch', 'Sa_Pa', 'là', 'khi', 'nào']
['Các', 'điểm', 'tham_quan', 'không', 'nên', 'bỏ', 'lỡ', 'khi', 'đến', 'Huế']
['Du_lịch', 'Ninh_Bình', 'nên', 'đi', 'Tràng_An', 'hay', 'Tam_Cốc']
['Địa_điểm', 'du_lịch', 'sinh_thái', 'nổi_bật', 'ở', 'miền', 'Tây_Nam_Bộ']
['Những', 'nơi', 'check', '-', 'in', 'đẹp', 'nhất', 'ở', 'Đà_Lạt', 'dành', 'cho', 'giới', 'trẻ']
['Du_lịch', 'Hạ_Long', 'có', 'những', 'tour', 'và', 'hoạt_động', 'nào', 'hấp_dẫn']
['Các', 'địa_điểm', 'du_lịch', 'tâm_linh', 'nổi_tiếng', 'ở', 'Việt_Nam']
['Nên', 'đi', 'du_lịch', 'Côn_Đảo', 'vào', 'mùa', 'nào', 'trong', 'năm']
['Các', 'địa_điểm', 'du_lịch', 'gần', 'TP', 'HCM', 'phù_hợp', 'đi', 'cuố

In [47]:
run

{'1': {'494': 0.9227422475814819,
  '362': 0.6725581288337708,
  '363': 0.6321331858634949,
  '57': 0.5669968128204346,
  '314': 0.377731591463089,
  '67': 0.35098573565483093,
  '495': 0.3450116515159607,
  '232': 0.20866039395332336,
  '78': 0.1399473398923874,
  '205': 0.1339460015296936,
  '201': 0.13261182606220245,
  '333': 0.13084493577480316,
  '172': 0.13066509366035461,
  '405': 0.12268981337547302,
  '207': 0.12227043509483337,
  '297': 0.10936786234378815,
  '372': 0.10127941519021988,
  '368': 0.09948353469371796,
  '349': 0.09234626591205597,
  '216': 0.09007930010557175,
  '410': 0.08935032784938812,
  '328': 0.08431138098239899,
  '390': 0.08277437090873718,
  '331': 0.08263891935348511,
  '400': 0.07916349172592163,
  '5': 0.07566401362419128,
  '167': 0.06895952671766281,
  '177': 0.06559230387210846,
  '221': 0.06547817587852478,
  '165': 0.06519781053066254,
  '392': 0.06406454741954803,
  '445': 0.061488546431064606,
  '49': 0.06024604290723801,
  '235': 0.05858768

In [48]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.3500
    Recall    : 0.8750
    F1        : 0.5000
------------------------------
Query 4
  @ 5
    Precision : 1.0000
    Recall    : 0.4167
    F1        : 0.5882
  @ 10
    Precision : 1.0000
    Recall    : 0.8333
    F1        : 0.9091
  @

## Cac term la 1 tu

In [49]:
def preprocess_query_word(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [50]:
import re

def preprocess_word(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    processed = []

    for t in tokens:
        t = t.lower()
        if len(t) <= 1:
            continue

        if "_" in t:
            processed.extend(t.split("_"))
        else:
            processed.append(t)

    return " ".join(processed)


In [51]:
import shutil
import os

def build_index_word(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = row["title"]
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess_word(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [52]:
def readQuery_word(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query_word(row["query"])

    return queries


In [53]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_word(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery_word(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã xây dựng xong LSA
✅ Đã chạy xong tất cả truy vấn


In [54]:
run

{'1': {'94': 0.37506645917892456,
  '93': 0.3466194272041321,
  '187': 0.2885112464427948,
  '89': 0.263169527053833,
  '92': 0.24873463809490204,
  '90': 0.2422967553138733,
  '131': 0.23571738600730896,
  '63': 0.22062866389751434,
  '51': 0.21617525815963745,
  '75': 0.21017546951770782,
  '58': 0.19828853011131287,
  '327': 0.19129899144172668,
  '114': 0.190904438495636,
  '17': 0.18579117953777313,
  '50': 0.17142696678638458,
  '15': 0.16192695498466492,
  '100': 0.15901149809360504,
  '350': 0.14651715755462646,
  '401': 0.13792741298675537,
  '95': 0.12983757257461548,
  '31': 0.1290753185749054,
  '186': 0.1265505850315094,
  '128': 0.12387102842330933,
  '10': 0.11607401072978973,
  '386': 0.11546199768781662,
  '368': 0.11390164494514465,
  '67': 0.1126900315284729,
  '409': 0.10848203301429749,
  '326': 0.10664597153663635,
  '133': 0.10389716923236847,
  '237': 0.10030823945999146,
  '167': 0.09394047409296036,
  '369': 0.09184874594211578,
  '224': 0.08999019861221313,
 

In [55]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 2
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0500
    Recall    : 0.1250
    F1        : 0.0714
------------------------------
Query 3
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 4
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.2000
    Recall    : 0.1667
    F1        : 0.1818
  @

In [57]:
print_best_worst_queries(run, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 33
Query    : đèo nào đẹp và nổi_tiếng để phượt ở miền trung
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['436', '437']
Top retrieved docs: ['437', '436', '100', '157', '26', '189', '64', '493', '68', '37']

Query ID : 7
Query    : các điểm tham_quan không nên bỏ lỡ khi đến huế
MAP      : 0.9750
P@10     : 0.8000
Relevant docs (8): ['114', '128', '248', '353', '355', '386', '436', '437']
Top retrieved docs: ['114', '386', '128', '437', '355', '353', '248', '193', '64', '436']

Query ID : 30
Query    : ruộng bậc_thang nào nổi_tiếng ở miền núi phía bắc
MAP      : 0.8333
P@10     : 0.2000
Relevant docs (2): ['308', '309']
Top retrieved docs: ['309', '70', '308', '126', '100', '383', '189', '86', '304', '111']

Query ID : 24
Query    : di_tích lịch_sử nào nổi_bật tại cố_đô huế
MAP      : 0.7222
P@10     : 0.3000
Relevant docs (3): ['114', '128', '248']
Top retrieved docs: ['114', '386', '128', '437', '353', '248', '355', '117', '64', '436

## Them PageRank va Quan trong

In [58]:
def normalize(scores):
    min_s, max_s = min(scores), max(scores)
    return [(s - min_s) / (max_s - min_s + 1e-9) for s in scores]

In [59]:
ALPHA = 0.8

def rank_with_pagerank(run, pagerank_dict):
    """
    run: dict[qid][doc_id] = lsa_score
    pagerank_dict: dict[doc_id] = pagerank
    """
    final_run = {}

    for qid, doc_scores in run.items():
        # lấy danh sách doc_id và score
        doc_ids = list(doc_scores.keys())
        lsa_scores = list(doc_scores.values())

        # lấy pagerank (ép kiểu nếu cần)
        pr_scores = [
            pagerank_dict.get(int(doc_id), 0.0)
            for doc_id in doc_ids
        ]

        # normalize pagerank theo query
        pr_norm = normalize(pr_scores)

        # kết hợp điểm
        reranked = {}
        for doc_id, lsa, pr in zip(doc_ids, lsa_scores, pr_norm):
            score = ALPHA * lsa + (1 - ALPHA) * pr
            reranked[doc_id] = score

        # sort giảm dần
        reranked = dict(
            sorted(reranked.items(), key=lambda x: x[1], reverse=True)
        )

        final_run[qid] = reranked

    return final_run


In [60]:
import pandas as pd

def load_pagerank(meta_csv):
    df = pd.read_csv(meta_csv)

    # giả sử cột là: doc_id, pagerank
    pagerank = dict(zip(df["id"], df["pagerank"]))

    return pagerank

In [62]:
pagerank = load_pagerank(META_CSV)

In [63]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# Load PageRank scores
pagerank_dict = load_pagerank(META_CSV)

# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")
final_run = rank_with_pagerank(run, pagerank_dict)
print("✅ Đã xếp hạng lại với PageRank")

✅ Đã xây dựng xong index
['Nhà_thờ', 'ở', 'Kon_Tum']
['Vũng_Tàu', 'có', 'các', 'địa_điểm', 'nào', 'đẹp']
['Những', 'địa_điểm', 'du_lịch', 'nổi_tiếng', 'nhất', 'ở', 'Hà_Nội', 'là', 'gì']
['Nên', 'đi', 'đâu', 'khi', 'du_lịch', 'Đà_Nẵng']
['Du_lịch', 'Hội_An', 'có', 'những', 'trải', 'nghiệm', 'đặc_sắc', 'nào']
['Thời_điểm', 'lý_tưởng', 'để', 'du_lịch', 'Sa_Pa', 'là', 'khi', 'nào']
['Các', 'điểm', 'tham_quan', 'không', 'nên', 'bỏ', 'lỡ', 'khi', 'đến', 'Huế']
['Du_lịch', 'Ninh_Bình', 'nên', 'đi', 'Tràng_An', 'hay', 'Tam_Cốc']
['Địa_điểm', 'du_lịch', 'sinh_thái', 'nổi_bật', 'ở', 'miền', 'Tây_Nam_Bộ']
['Những', 'nơi', 'check', '-', 'in', 'đẹp', 'nhất', 'ở', 'Đà_Lạt', 'dành', 'cho', 'giới', 'trẻ']
['Du_lịch', 'Hạ_Long', 'có', 'những', 'tour', 'và', 'hoạt_động', 'nào', 'hấp_dẫn']
['Các', 'địa_điểm', 'du_lịch', 'tâm_linh', 'nổi_tiếng', 'ở', 'Việt_Nam']
['Nên', 'đi', 'du_lịch', 'Côn_Đảo', 'vào', 'mùa', 'nào', 'trong', 'năm']
['Các', 'địa_điểm', 'du_lịch', 'gần', 'TP', 'HCM', 'phù_hợp', 'đi', 'cuố

In [64]:
final_run

{'1': {'494': 0.7387074947357178,
  '362': 0.5345952033996583,
  '57': 0.5315200003620094,
  '363': 0.504665470123291,
  '67': 0.3637651833530373,
  '314': 0.30297479629516605,
  '495': 0.27318634986877444,
  '232': 0.2578965138542653,
  '78': 0.23546112128496507,
  '109': 0.2239416563757137,
  '95': 0.18916869913355555,
  '49': 0.17335108224461565,
  '150': 0.16250697269535916,
  '297': 0.1593561827670421,
  '235': 0.1571955794954279,
  '90': 0.15689282366037705,
  '205': 0.1554798858411177,
  '129': 0.15546166540939027,
  '37': 0.15287729450464585,
  '177': 0.1522741368420994,
  '114': 0.15160441347360948,
  '207': 0.14625848701070385,
  '172': 0.14408780814187924,
  '124': 0.13921660797602542,
  '221': 0.13912213768920365,
  '165': 0.13589163660031878,
  '113': 0.12673815931283872,
  '16': 0.12206061508616868,
  '224': 0.12012392606696548,
  '333': 0.1147268295288086,
  '201': 0.11409001002183157,
  '313': 0.10961855595782707,
  '47': 0.10940899875125351,
  '167': 0.1067596055152842

In [65]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    final_run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.2000
    Recall    : 0.1250
    F1        : 0.1538
  @ 10
    Precision : 0.3000
    Recall    : 0.3750
    F1        : 0.3333
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 4
  @ 5
    Precision : 1.0000
    Recall    : 0.4167
    F1        : 0.5882
  @ 10
    Precision : 0.8000
    Recall    : 0.6667
    F1        : 0.7273
  @

In [66]:
final_run

{'1': {'494': 0.7387074947357178,
  '362': 0.5345952033996583,
  '57': 0.5315200003620094,
  '363': 0.504665470123291,
  '67': 0.3637651833530373,
  '314': 0.30297479629516605,
  '495': 0.27318634986877444,
  '232': 0.2578965138542653,
  '78': 0.23546112128496507,
  '109': 0.2239416563757137,
  '95': 0.18916869913355555,
  '49': 0.17335108224461565,
  '150': 0.16250697269535916,
  '297': 0.1593561827670421,
  '235': 0.1571955794954279,
  '90': 0.15689282366037705,
  '205': 0.1554798858411177,
  '129': 0.15546166540939027,
  '37': 0.15287729450464585,
  '177': 0.1522741368420994,
  '114': 0.15160441347360948,
  '207': 0.14625848701070385,
  '172': 0.14408780814187924,
  '124': 0.13921660797602542,
  '221': 0.13912213768920365,
  '165': 0.13589163660031878,
  '113': 0.12673815931283872,
  '16': 0.12206061508616868,
  '224': 0.12012392606696548,
  '333': 0.1147268295288086,
  '201': 0.11409001002183157,
  '313': 0.10961855595782707,
  '47': 0.10940899875125351,
  '167': 0.1067596055152842

In [67]:
print_best_worst_queries(run, queries, qrels)


🔥 TOP QUERIES (Highest MAP)

Query ID : 11
Query    : du_lịch hạ_long tour hoạt_động hấp_dẫn
MAP      : 1.0000
P@10     : 0.4000
Relevant docs (4): ['50', '122', '132', '133']
Top retrieved docs: ['133', '50', '122', '132', '93', '229', '203', '121', '168', '338']

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['30', '218', '169', '168', '376', '331', '119', '377', '216', '349']

Query ID : 27
Query    : chợ đêm nổi_tiếng đông đà_lạt
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['143', '38']
Top retrieved docs: ['38', '143', '363', '217', '172', '275', '362', '365', '318', '216']

Query ID : 33
Query    : đèo đẹp nổi_tiếng phượt miền trung
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['436', '437']
Top retrieved docs: ['437', '436', '100', '37', '64', '189', '61', '493', '68', '52']

Query ID : 37
Query    : thành nhà_hồ
MAP      : 1.0000
P@10     : 0.1000
Relevant docs (1): ['151']
To